# 02 · Catalog EDA

One row per game (`game_catalog()`, cached): coverage of each attribute, distributions, and which
genres / tags go with positive reviews.

In [ ]:
import os

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

from steam_eda import (
    Sample,
    binned_rate,
    cached,
    categorical_rate,
    decode,
    feature_report,
    game_catalog,
    gini,
    iter_mart,
    load_mart,
    load_raw,
    lookup,
    lorenz,
    mart_schema,
    mart_table,
    raw_files,
    review_counts,
    tag_profile_similarity,
    with_user_history,
)
from steam_eda.plots import loglog_hist, rate_plot, setup

setup()

SAMPLE_ROWS = float(os.environ.get("EDA_SAMPLE_ROWS", 0.01))  # interactions for rates by attribute
catalog = cached("catalog", game_catalog)
catalog.height

## Coverage

In [ ]:
catalog.select(
    pl.len().alias("games"),
    (pl.col("reviews") > 0).mean().alias("reviewed"),
    pl.col("game_coming_soon").fill_null(False).mean().alias("coming soon"),
    pl.col("game_is_free").fill_null(False).mean().alias("free"),
    pl.col("game_price").is_not_null().mean().alias("price known"),
    pl.col("release_year").is_not_null().mean().alias("release year parsed"),
    (pl.col("game_tags").list.len() > 0).mean().alias("has tags"),
    (pl.col("game_genres").list.len() > 0).mean().alias("has genres"),
    (pl.col("game_developers").list.len() > 0).mean().alias("has developer"),
    pl.col("game_short_description").is_not_null().mean().alias("has description"),
).transpose(include_header=True, column_names=["share"])

## Distributions (reviewed, released games)

In [ ]:
released = catalog.filter((pl.col("reviews") > 0) & ~pl.col("game_coming_soon").fill_null(False))
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
sns.histplot(released["game_reviews_ratio"].to_pandas(), bins=50, ax=axes[0, 0]).set_title(
    "reviews ratio (smoothed)"
)
sns.histplot(
    released.filter(pl.col("game_price") > 0)["game_price"].clip(upper_bound=80).to_pandas(),
    bins=80,
    ax=axes[0, 1],
).set_title("price (paid, clipped at 80)")
years = (
    released.group_by("release_year")
    .agg(games=pl.len(), reviews=pl.col("reviews").sum())
    .drop_nulls()
    .sort("release_year")
)
axes[1, 0].bar(years["release_year"], years["games"])
axes[1, 0].set_title("games by release year")
axes[1, 1].bar(years["release_year"], years["reviews"])
axes[1, 1].set_title("reviews by release year")
plt.tight_layout()

## Tags: vocabulary, per-game counts, concentration

In [ ]:
tag_games = (
    catalog.select("game_idx", "game_tags", "reviews")
    .explode("game_tags", empty_as_null=False)
    .rename({"game_tags": "id"})
    .group_by("id")
    .agg(games=pl.len(), reviews=pl.col("reviews").sum())
    .join(lookup("lkp_tags"), on="id")
    .sort("games", descending=True)
)
print(
    f"{tag_games.height} tags; tags per game: "
    f"{catalog['game_tags'].list.len().describe().filter(pl.col('statistic').is_in(['mean', '50%', 'max']))}"
)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.barplot(tag_games.head(25).to_pandas(), y="name", x="games", ax=axes[0]).set_title(
    "top tags by games"
)
sns.barplot(
    tag_games.sort("reviews", descending=True).head(25).to_pandas(),
    y="name",
    x="reviews",
    ax=axes[1],
).set_title("top tags by reviews")
plt.tight_layout()
tag_games.select((pl.col("games") < 50).sum().alias("tags on < 50 games"))

## Positive rate by attribute

A tag / genre with a rate far from the average is item-side signal (the reviews ratio already
carries part of it per game).

In [ ]:
sample = load_mart(
    "interactions", columns=["game_idx", "is_positive"], sample=Sample.rows(SAMPLE_ROWS)
)
sample = sample.join(
    catalog.select("game_idx", "game_genres", "game_tags", "game_is_free", "release_year"),
    on="game_idx",
    how="left",
)
by_tag = categorical_rate(sample, "game_tags", min_rows=2_000).join(
    lookup("lkp_tags"), left_on="game_tags", right_on="id"
)
by_genre = categorical_rate(sample, "game_genres", min_rows=2_000).join(
    lookup("lkp_genres"), left_on="game_genres", right_on="id"
)
print(f"{sample.height:,d} sampled interactions, positive rate {sample['is_positive'].mean():.3f}")
pl.concat([by_tag.head(10), by_tag.tail(10)]).select("name", "rows", "rate", "lift")

In [ ]:
by_genre.select("name", "rows", "rate", "lift")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
rate_plot(binned_rate(sample, "release_year", bins=15), "positive rate by release year", ax=axes[0])
rate_plot(
    categorical_rate(sample, "game_is_free", min_rows=1).sort("game_is_free"),
    "positive rate free vs paid",
    x="game_is_free",
    ax=axes[1],
)

## Notes

Coverage gaps, skews (release years, tags), attributes with lift.